### **ChromaDB**

#### **Difference between FAISS and ChromaDB**

ChromaDB is not a simple Vectorstore, it is actually a fully-fledged VectorDB. In local it has some limitations, but in the cloud variant, it is a full-fledged solution. 


faiss is a library provided by Meta. Now on top of it, we have a LangChain FAISS wrapper to convert it into a vectorstore. Originally faiss only stores embeddings, but with FAISS wrapper, we can store metadata, document ID, etc. as in a vectorstore. 

| Feature                | FAISS   | ChromaDB |
| ---------------------- | ------- | -------- |
| Flat index             | ✅       | ❌        |
| IVF                    | ✅       | ❌        |
| HNSW                   | ✅       | ✅        |
| PQ (Product Quantisation)                     | ✅       | ❌        |
| Manual index selection | ✅       | ❌        |
| Metadata filtering     | Limited | ✅        |
| Persistence            | Manual  | Built-in |
| LangChain integration  | ✅       | ✅        |


Final understanding
FAISS = vector index/search engine

You manually manage:
- index type
- dimension
- metric
- docstore
- ID mapping
- persistence
  
Chroma = vector database

It manages the following on its own:
- vectors
- documents
- metadata
- IDs
- collections
- index
- persistence


FAISS is a low-level, high-performance library for dense-vector similarity search and clustering. It gives developers direct control over index structures such as Flat, IVF, HNSW and product-quantized indexes, and it offers strong CPU and GPU capabilities. However, FAISS is not a complete vector database: document storage, metadata management, filtering, CRUD APIs, collections, persistence orchestration and server infrastructure generally need to be handled separately.

Chroma is a retrieval database/search infrastructure designed for AI applications. It stores embeddings together with documents, metadata and IDs, and provides collections, persistence, metadata filtering, full-text and sparse retrieval, CRUD operations and client-server or hosted deployment options. In current Chroma, single-node vector search uses HNSW, while its broader schema and cloud architecture also support other retrieval indexes such as SPANN and sparse/full-text indexes.

Therefore, FAISS is preferable when low-level index control, custom ANN algorithms, compression or GPU optimization is the priority. Chroma is preferable when building a complete RAG application that needs database-style storage, filtering, updates and operational simplicity.

| Feature         | FAISS                 | Chroma |
| --------------- | --------------------- | ------ |
| Store vectors   | ✅                     | ✅      |
| Store documents | ❌                     | ✅      |
| Store metadata  | ❌                     | ✅      |
| Collections     | ❌                     | ✅      |
| CRUD            | Limited               | ✅      |
| Filtering       | ❌ (native)            | ✅      |
| Persistence     | Basic index save/load | ✅      |
| Client APIs     | ❌                     | ✅      |
| Server mode     | ❌                     | ✅      |


Tumhare code me FAISS ke saath document aur metadata dono store ho rahe the, but crucial point ye hai:

Unhe native FAISS store nahi kar raha tha; LangChain ka FAISS wrapper store kar raha tha.

LangChain FAISS VectorStore
│
├── FAISS index
│     └── Numerical embedding vectors
│
├── InMemoryDocstore
│     └── LangChain Document objects
│         ├── page_content
│         └── metadata
│
└── index_to_docstore_id
      └── FAISS position ko Document ID se map karta hai

vector_store = FAISS(
    embedding_function=embeddings,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

Chroma me document, metadata, ID aur embedding same database collection ke records hain:

Chroma collection

collection.add(
    ids=["chunk-1"],
    documents=["Llama 2 is a family of language models."],
    metadatas=[
        {
            "source": "llama2.pdf",
            "page": 5
        }
    ],
    embeddings=[[0.1, 0.2, 0.3]]
)

| Component        | LangChain + FAISS             | Chroma                          |
| ---------------- | ----------------------------- | ------------------------------- |
| Embeddings       | Native FAISS index            | Chroma vector index             |
| Documents        | LangChain `Docstore`          | Chroma collection               |
| Metadata         | LangChain `Document.metadata` | Chroma collection record        |
| Mapping          | `index_to_docstore_id`        | Internally managed              |
| Save             | FAISS file + pickle           | Database persistence            |
| Metadata filters | Wrapper/application handling  | Native database filtering       |
| Collections      | Not native to FAISS           | Native                          |
| CRUD             | Wrapper/index-dependent       | Native record operations        |
| Server/cloud     | Separate system required      | Supported database architecture |


https://www.trychroma.com/

#### **Working with ChromaDB**

##### **Imports**

In [1]:
from dotenv import load_dotenv
import os

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

C:\Users\HP\AppData\Local\Temp\ipykernel_23844\3536989817.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")

##### **Loading the PDF**

In [13]:
file_path = os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), "Research//LLAMA2_research_paper.pdf")
if os.path.exists(file_path):    
    loader = PyPDFLoader(file_path)
    pages = loader.load()
    print("Total pages:", len(pages))


Total pages: 77


##### **Creating Chunks**

In [14]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)
chunks = text_splitter.split_documents(pages)
print("Total chunks:", len(chunks))

Total chunks: 175


##### **Creating Chroma Vectorstore**

In [15]:
vector_store = Chroma(
    collection_name="llama2_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_llama2",
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

##### **Storing Data inside the Vectorstore**

In [16]:
document_ids = vector_store.add_documents(documents=chunks)

print("Documents added:", len(document_ids))

print("Total documents stored:",vector_store._collection.count())

Documents added: 175
Total documents stored: 175


##### **Create Retriever**

In [17]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

##### **Retrieval**

In [18]:
query = "What is the architecture of Llama 2?"

retrieved_documents = retriever.invoke(query)

for i, document in enumerate(
    retrieved_documents,
    start=1
):
    print(f"\n--- Retrieved document {i} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)



--- Retrieved document 1 ---
A.7 Model Card
Table 52 presents a model card (Mitchell et al., 2018; Anil et al., 2023) that summarizes details of the models.
Model Details
Model DevelopersMeta AI
Variations Llama 2comes in a range of parameter sizes—7B, 13B, and 70B—as well as
pretrained and fine-tuned variations.
Input Models input text only.
Output Models generate text only.
Model ArchitectureLlama 2isanauto-regressivelanguagemodelthatusesanoptimizedtransformer
architecture. The tuned versions use supervised fine-tuning (S
Metadata: {'source': 'c:\\Data_science\\Modern_route\\Research//LLAMA2_research_paper.pdf', 'keywords': '', 'author': '', 'title': '', 'page': 76, 'creationdate': '2023-07-20T00:30:36+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'producer': 'pdfTeX-1.40.25', 'total_pages': 77, 'moddate': '2023-07-20T00:30:36+00:00', 'page_label': '77', 'creator': 'LaTeX with 

##### **Prompt**

In [19]:
prompt = ChatPromptTemplate.from_template(
    """
    You are a question-answering assistant.

    Answer the question only from the provided context.

    If the context does not contain the answer, say:
    "I do not have enough information in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

##### **Formatting the docs**

In [20]:
def format_docs(docs):
    return "\n\n".join(
        f"""
        Source: {doc.metadata.get("source")}
        Page: {doc.metadata.get("page")}

        {doc.page_content}
        """
        for doc in docs
    )


##### **LLM Call**

In [21]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

##### **RAG Chain**

In [22]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

##### **Question & answer**

In [23]:
answer = rag_chain.invoke(
    "What is the architecture of Llama 2?"
)

print("\nFinal answer:\n")
print(answer)


Final answer:

Llama 2 is an auto-regressive language model that uses an optimized transformer architecture. It adopts most of the pretraining setting and model architecture from Llama 1, using the standard transformer architecture, applying pre-normalization using RMSNorm, using the SwiGLU activation function, and rotary positional embeddings (RoPE). The primary architectural differences from Llama 1 include increased context length and grouped-query attention (GQA). The tuned versions use supervised fine-tuning (SFT) and reinforcement learning with human feedback (RLHF) to align to human preferences for helpfulness and safety.
